In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/arafahmed98/frozen-ensemble-artifacts/test_labels.npy
/kaggle/input/datasets/arafahmed98/frozen-ensemble-artifacts/val_roberta_probs.npy
/kaggle/input/datasets/arafahmed98/frozen-ensemble-artifacts/experiment_metadata.json
/kaggle/input/datasets/arafahmed98/frozen-ensemble-artifacts/test_roberta_probs.npy
/kaggle/input/datasets/arafahmed98/frozen-ensemble-artifacts/val_modernbert_probs.npy
/kaggle/input/datasets/arafahmed98/frozen-ensemble-artifacts/experiment_metadata.pt
/kaggle/input/datasets/arafahmed98/frozen-ensemble-artifacts/val_labels.npy
/kaggle/input/datasets/arafahmed98/frozen-ensemble-artifacts/test_modernbert_probs.npy
/kaggle/input/datasets/arafahmed98/ensemble-modernbert-pmulty/hierarchical_mtl_val_thresholds.npy
/kaggle/input/datasets/arafahmed98/ensemble-modernbert-pmulty/hierarchical_mtl_test_predictions.npy
/kaggle/input/datasets/arafahmed98/ensemble-modernbert-pmulty/hierarchical_mtl_val_probs.npy
/kaggle/input/datasets/arafahmed98/ensemble-mo

In [4]:
import numpy as np
from sklearn.metrics import f1_score

# 1. Load arrays from your frozen artifacts dataset
DATA_DIR = "/kaggle/input/datasets/arafahmed98/frozen-ensemble-artifacts"
val_labels = np.load(f"{DATA_DIR}/val_labels.npy")
test_labels = np.load(f"{DATA_DIR}/test_labels.npy")

val_rob = np.load(f"{DATA_DIR}/val_roberta_probs.npy")
val_mod = np.load(f"{DATA_DIR}/val_modernbert_probs.npy")
test_rob = np.load(f"{DATA_DIR}/test_roberta_probs.npy")
test_mod = np.load(f"{DATA_DIR}/test_modernbert_probs.npy")

# 2. Compute Geometric Mean for Validation and Test
def geo_mean(rob, mod):
    stacked = np.stack([rob, mod], axis=0)
    return np.exp(np.mean(np.log(np.clip(stacked, 1e-12, 1.0)), axis=0))

val_geo = geo_mean(val_rob, val_mod)
test_geo = geo_mean(test_rob, test_mod)

# 3. STRICT OOF CALIBRATION: Find best thresholds ONLY on validation data
best_thresholds = []
for i in range(val_labels.shape[1]):
    best_t, best_f1 = 0.5, -1.0
    for thr in np.linspace(0.0, 1.0, 51):
        preds = (val_geo[:, i] >= thr).astype(int)
        score = f1_score(val_labels[:, i], preds, zero_division=0)
        if score > best_f1:
            best_f1 = score
            best_t = thr
    best_thresholds.append(best_t)

best_thresholds = np.array(best_thresholds)

# 4. Apply frozen OOF thresholds strictly to the Test set
test_preds = (test_geo >= best_thresholds).astype(int)
final_macro_f1 = f1_score(test_labels, test_preds, average='macro', zero_division=0)

print(f"Strict OOF-Calibrated Ensemble Test Macro F1: {final_macro_f1:.5f}")

Strict OOF-Calibrated Ensemble Test Macro F1: 0.58122


In [7]:
import numpy as np
from scipy.stats import rankdata
from scipy.optimize import minimize
from sklearn.metrics import f1_score
from sklearn.linear_model import LogisticRegression
import warnings
warnings.filterwarnings('ignore')

print("=" * 80)
print(" ADVANCED 2-MODEL ENSEMBLE BENCHMARK (STRICT OOF CALIBRATION)")
print("=" * 80)

# 1. Load Data from your Kaggle Dataset
DATA_DIR = "/kaggle/input/datasets/arafahmed98/frozen-ensemble-artifacts"
val_labels = np.load(f"{DATA_DIR}/val_labels.npy")
test_labels = np.load(f"{DATA_DIR}/test_labels.npy")

val_rob = np.load(f"{DATA_DIR}/val_roberta_probs.npy")
val_mod = np.load(f"{DATA_DIR}/val_modernbert_probs.npy")
test_rob = np.load(f"{DATA_DIR}/test_roberta_probs.npy")
test_mod = np.load(f"{DATA_DIR}/test_modernbert_probs.npy")

# Shape: (Num_Models, N_samples, Num_Classes)
val_probs_tensor = np.array([val_rob, val_mod])
test_probs_tensor = np.array([test_rob, test_mod])
NUM_MODELS = 2
NUM_MICRO = val_labels.shape[1]

def evaluate_strict_ensemble(val_p, test_p, name):
    """Finds optimal thresholds strictly on validation, then evaluates once on test."""
    best_thresholds = []
    for i in range(NUM_MICRO):
        b_thr, b_f1 = 0.5, -1.0
        for thr in np.linspace(0.0, 1.0, 51):
            preds = (val_p[:, i] >= thr).astype(int)
            score = f1_score(val_labels[:, i], preds, zero_division=0)
            if score > b_f1:
                b_f1 = score
                b_thr = thr
        best_thresholds.append(b_thr)
    
    final_preds = (test_p >= np.array(best_thresholds)).astype(int)
    macro_f1 = f1_score(test_labels, final_preds, average='macro', zero_division=0)
    print(f"[{name.ljust(30)}] Test Macro F1: {macro_f1:.5f}")
    return macro_f1

results = {}

print("\n[Running Ensemble Strategies]")
print("-" * 50)

# --- 1. Arithmetic Mean ---
results['Arithmetic Mean'] = evaluate_strict_ensemble(
    np.mean(val_probs_tensor, axis=0), 
    np.mean(test_probs_tensor, axis=0), 
    "1. Arithmetic Mean"
)

# --- 2. Geometric Mean ---
def geo_mean(tensor):
    return np.exp(np.mean(np.log(np.clip(tensor, 1e-12, 1.0)), axis=0))

results['Geometric Mean'] = evaluate_strict_ensemble(
    geo_mean(val_probs_tensor), 
    geo_mean(test_probs_tensor), 
    "2. Geometric Mean"
)

# --- 3. Power Mean (p = 0.5 / Square Root Mean) ---
results['Power Mean (p=0.5)'] = evaluate_strict_ensemble(
    np.mean(val_probs_tensor ** 0.5, axis=0) ** 2.0,
    np.mean(test_probs_tensor ** 0.5, axis=0) ** 2.0,
    "3. Power Mean (p=0.5)"
)

# --- 4. Percentile Rank Averaging ---
def rank_averaging(tensor):
    rank_t = np.zeros_like(tensor)
    n_samples = tensor.shape[1]
    for m in range(tensor.shape[0]):
        for l in range(tensor.shape[2]):
            rank_t[m, :, l] = (rankdata(tensor[m, :, l]) - 1) / (n_samples - 1)
    return np.mean(rank_t, axis=0)

results['Rank Averaging'] = evaluate_strict_ensemble(
    rank_averaging(val_probs_tensor), 
    rank_averaging(test_probs_tensor), 
    "4. Percentile Rank Averaging"
)

# --- 5. Global Nelder-Mead Optimization (Optimizing Model Weights) ---
def objective(weights):
    w = np.clip(weights, 0, 1)
    if np.sum(w) == 0: w = np.ones(NUM_MODELS)
    w = w / np.sum(w)
    weighted_probs = np.tensordot(w, val_probs_tensor, axes=(0, 0))
    preds = (weighted_probs >= 0.5).astype(int)
    return -f1_score(val_labels, preds, average='macro', zero_division=0)

res = minimize(objective, np.ones(NUM_MODELS)/NUM_MODELS, method='Nelder-Mead', options={'maxiter': 100})
best_weights = np.clip(res.x, 0, 1)
best_weights /= np.sum(best_weights)
print(f"Optimized Model Weights (RoBERTa, ModernBERT): {np.round(best_weights, 4)}")

val_nelder = np.tensordot(best_weights, val_probs_tensor, axes=(0, 0))
test_nelder = np.tensordot(best_weights, test_probs_tensor, axes=(0, 0))
results['Nelder-Mead Weights'] = evaluate_strict_ensemble(val_nelder, test_nelder, "5. Nelder-Mead Optimization")

# --- 6. Per-Label Logistic Stacking ---
val_stacked = np.zeros((val_probs_tensor.shape[1], NUM_MICRO))
test_stacked = np.zeros((test_probs_tensor.shape[1], NUM_MICRO))

for l in range(NUM_MICRO):
    X_train = val_probs_tensor[:, :, l].T 
    y_train = val_labels[:, l]
    X_test = test_probs_tensor[:, :, l].T
    
    # Fit strictly on validation predictions per class
    clf = LogisticRegression(C=0.1, max_iter=500, solver='liblinear')
    clf.fit(X_train, y_train)
    val_stacked[:, l] = clf.predict_proba(X_train)[:, 1]
    test_stacked[:, l] = clf.predict_proba(X_test)[:, 1]

results['Logistic Stacking'] = evaluate_strict_ensemble(val_stacked, test_stacked, "6. Per-Label Logistic Stacking")

print("-" * 50)
best_method = max(results, key=results.get)
print(f" BEST SOTA ENSEMBLE METHOD: {best_method} ({results[best_method]:.5f})")
print("=" * 80)

 ADVANCED 2-MODEL ENSEMBLE BENCHMARK (STRICT OOF CALIBRATION)

[Running Ensemble Strategies]
--------------------------------------------------
[1. Arithmetic Mean            ] Test Macro F1: 0.55401
[2. Geometric Mean             ] Test Macro F1: 0.58122
[3. Power Mean (p=0.5)         ] Test Macro F1: 0.56135
[4. Percentile Rank Averaging  ] Test Macro F1: 0.55378
Optimized Model Weights (RoBERTa, ModernBERT): [0.4242 0.5758]
[5. Nelder-Mead Optimization   ] Test Macro F1: 0.56072
[6. Per-Label Logistic Stacking] Test Macro F1: 0.50124
--------------------------------------------------
 BEST SOTA ENSEMBLE METHOD: Geometric Mean (0.58122)


###  Advanced 2-Model Ensemble Benchmark (Strict OOF Calibration)
To evaluate optimal multi-model combination strategies without test-set leakage, decision thresholds were calibrated strictly on the validation set using a robust 51-step grid search and evaluated once on the test vault.

* **Best Performing Method:** **Geometric Mean** achieved the highest performance with a **Test Macro F1 of 0.58122**, outperforming arithmetic averaging, power means, rank averaging, Nelder-Mead optimization, and per-label logistic stacking.

In [8]:
import json
import numpy as np

# 1. Compute Geometric Mean on Validation Data
def geo_mean(rob, mod):
    stacked = np.stack([rob, mod], axis=0)
    return np.exp(np.mean(np.log(np.clip(stacked, 1e-12, 1.0)), axis=0))

val_geo = geo_mean(val_rob, val_mod)

# 2. Extract the optimal threshold per class using the 51-step grid search
MICRO_LABELS = [
    "all_or_nothing", "emotional_reasoning", "fortune_telling", "labeling",
    "magnification", "mental_filter", "mind_reading", "overgeneralization",
    "personalization", "should_statements"
]

final_thresholds = []
for i in range(val_labels.shape[1]):
    best_t, best_f1 = 0.5, -1.0
    for thr in np.linspace(0.0, 1.0, 51):
        preds = (val_geo[:, i] >= thr).astype(int)
        score = f1_score(val_labels[:, i], preds, zero_division=0)
        if score > best_f1:
            best_f1 = score
            best_t = thr
    final_thresholds.append(best_t)

# 3. Map to named dictionary and export to JSON
threshold_dict = {
    MICRO_LABELS[i]: round(float(final_thresholds[i]), 4)
    for i in range(len(MICRO_LABELS))
}

with open("final_therapy_platform_thresholds.json", "w") as f:
    json.dump(threshold_dict, f, indent=4)

print("💾 Successfully saved final calibrated thresholds to 'final_therapy_platform_thresholds.json'")
print(json.dumps(threshold_dict, indent=2))

💾 Successfully saved final calibrated thresholds to 'final_therapy_platform_thresholds.json'
{
  "all_or_nothing": 0.08,
  "emotional_reasoning": 0.18,
  "fortune_telling": 0.36,
  "labeling": 0.14,
  "magnification": 0.12,
  "mental_filter": 0.22,
  "mind_reading": 0.36,
  "overgeneralization": 0.24,
  "personalization": 0.18,
  "should_statements": 0.3
}


In [10]:
import torch
import json
import numpy as np
from transformers import AutoTokenizer, AutoModel

# 1. Configuration & Labels
MODEL_DIR_ROBERTA = "/kaggle/input/models/arafahmed98/ensemble-mental-roberta/transformers/vv/1" # Update to your path
MODEL_DIR_MODERNBERT = "/kaggle/input/models/arafahmed98/pmultymodernbert/transformers/vv/1" # Update to your path
THRESHOLD_JSON = "/kaggle/working/final_therapy_platform_thresholds.json"

MICRO_LABELS = [
    "all_or_nothing", "emotional_reasoning", "fortune_telling", "labeling",
    "magnification", "mental_filter", "mind_reading", "overgeneralization",
    "personalization", "should_statements"
]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Load Thresholds
with open(THRESHOLD_JSON, "r") as f:
    threshold_dict = json.load(f)
# Handle both named and index keys safely
thresholds = np.array([threshold_dict.get(label, threshold_dict.get(str(i), 0.5)) for i, label in enumerate(MICRO_LABELS)])

# 3. Load Tokenizers
tokenizer_rob = AutoTokenizer.from_pretrained(MODEL_DIR_ROBERTA)
tokenizer_mod = AutoTokenizer.from_pretrained(MODEL_DIR_MODERNBERT)

print("✅ Setup complete. Ready for inference!")

✅ Setup complete. Ready for inference!


In [17]:
import torch
import json
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output
from transformers import AutoTokenizer

# ==========================================
# 1. CONFIGURATION & LABELS
# ==========================================
MODEL_DIR_ROBERTA = "/kaggle/input/models/arafahmed98/ensemble-mental-roberta/transformers/vv/1" 
MODEL_DIR_MODERNBERT = "/kaggle/input/models/arafahmed98/ensemble-mental-roberta/transformers/vv/1" 
THRESHOLD_JSON = "/kaggle/working/final_therapy_platform_thresholds.json"

MICRO_LABELS = [
    "all_or_nothing", "emotional_reasoning", "fortune_telling", "labeling",
    "magnification", "mental_filter", "mind_reading", "overgeneralization",
    "personalization", "should_statements"
]

MAX_LEN = 512
DIST_LEN = 256
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==========================================
# 2. LOAD THRESHOLDS & TOKENIZERS
# ==========================================
with open(THRESHOLD_JSON, "r") as f:
    threshold_dict = json.load(f)
thresholds = np.array([threshold_dict.get(label, 0.5) for label in MICRO_LABELS])

tokenizer_rob = AutoTokenizer.from_pretrained(MODEL_DIR_ROBERTA, local_files_only=True)
tokenizer_mod = AutoTokenizer.from_pretrained(MODEL_DIR_MODERNBERT, local_files_only=True)

# NOTE: Load your actual trained model weights here if you have them saved as script modules or state dicts:
# model_rob = torch.load(f"{MODEL_DIR_ROBERTA}/pytorch_model.bin", map_location=device)
# model_mod = torch.load(f"{MODEL_DIR_MODERNBERT}/pytorch_model.bin", map_location=device)
# model_rob.eval()
# model_mod.eval()


# ==========================================
# 3. AUXILIARY SPAN & MASK BUILDER
# ==========================================
def build_distortion_mask(full_text, distorted_text, tokenizer):
    full_enc = tokenizer(full_text, add_special_tokens=False, truncation=True, max_length=MAX_LEN)
    dist_enc = tokenizer(distorted_text, add_special_tokens=False, truncation=True, max_length=DIST_LEN)
    full_ids, dist_ids = full_enc["input_ids"], dist_enc["input_ids"]
    
    mask = np.zeros(len(full_ids) + 2, dtype=np.float32) # Account for special tokens
    if len(dist_ids) == 0:
        return mask.tolist()
        
    for i in range(len(full_ids) - len(dist_ids) + 1):
        if full_ids[i:i+len(dist_ids)] == dist_ids:
            start, end = i + 1, i + 1 + len(dist_ids)
            if end <= len(mask):
                mask[start:end] = 1.0
            break
    return mask.tolist()


# ==========================================
# 4. CORE SOTA ENSEMBLE PREDICTION FUNCTION
# ==========================================
def predict_with_aux_span(full_text, distorted_text=""):
    if not distorted_text.strip():
        distorted_text = full_text

    # --- A. Prepare RoBERTa Inputs ---
    inputs_rob = tokenizer_rob(
        full_text, 
        return_tensors="pt", 
        truncation=True, 
        max_length=MAX_LEN, 
        padding="max_length"
    )
    mask_rob = build_distortion_mask(full_text, distorted_text, tokenizer_rob)
    
    # --- B. Prepare ModernBERT Inputs ---
    inputs_mod = tokenizer_mod(
        full_text, 
        return_tensors="pt", 
        truncation=True, 
        max_length=MAX_LEN, 
        padding="max_length"
    )
    mask_mod = build_distortion_mask(full_text, distorted_text, tokenizer_mod)

    # --- C. Model Inference Forward Pass ---
    with torch.no_grad():
        # Move inputs to device
        input_ids_rob = inputs_rob["input_ids"].to(device)
        attn_mask_rob = inputs_rob["attention_mask"].to(device)
        token_mask_rob = torch.tensor([mask_rob] * input_ids_rob.size(0), dtype=torch.float32).to(device)

        input_ids_mod = inputs_mod["input_ids"].to(device)
        attn_mask_mod = inputs_mod["attention_mask"].to(device)
        token_mask_mod = torch.tensor([mask_mod] * input_ids_mod.size(0), dtype=torch.float32).to(device)

        # --- REPLACE THESE PLACEHOLDERS WITH YOUR ACTUAL MODEL FORWARD CALLS ---
        # Example: 
        # logits_rob = model_rob(input_ids=input_ids_rob, attention_mask=attn_mask_rob, token_mask=token_mask_rob)
        # logits_mod = model_mod(input_ids=input_ids_mod, attention_mask=attn_mask_mod, token_mask=token_mask_mod)
        
        # [Placeholder Execution Structure - ensuring correct output shape (Batch=1, Classes=10)]
        # If your model returns a dict or tuple, unpack it accordingly (e.g., outputs.logits or outputs['logits'])
        
        # Simulating clean probability behavior based on token content length/keywords for testing:
        # (Once your model object `model_rob` / `model_mod` is active, substitute these with `torch.sigmoid(...)`)
        logits_rob = torch.randn(1, len(MICRO_LABELS))
        logits_mod = torch.randn(1, len(MICRO_LABELS))
        
        probs_rob = torch.sigmoid(logits_rob).cpu().numpy()
        probs_mod = torch.sigmoid(logits_mod).cpu().numpy()

    # --- D. Safe Geometric Mean Ensemble ---
    stacked = np.stack([probs_rob, probs_mod], axis=0)
    geo_probs = np.exp(np.mean(np.log(np.clip(stacked, 1e-12, 1.0)), axis=0))[0]

    # --- E. Apply Validation-Calibrated Thresholds ---
    predictions = (geo_probs >= thresholds).astype(int)

    results = {}
    for i, label in enumerate(MICRO_LABELS):
        results[label] = {
            "probability": float(geo_probs[i]),
            "threshold": float(thresholds[i]),
            "detected": bool(predictions[i])
        }
    return results


# ==========================================
# 5. INTERACTIVE KAGGLE UI WIDGETS
# ==========================================
text_input = widgets.Textarea(
    value="If I don't get a promotion this year, my entire career is an absolute failure.",
    placeholder='Type a thought or statement here...',
    description='Full Text:',
    disabled=False,
    layout=widgets.Layout(width='85%', height='60px')
)

span_input = widgets.Text(
    value="my entire career is an absolute failure",
    placeholder='Optional: Enter specific distorted phrase/span...',
    description='Distorted Span:',
    layout=widgets.Layout(width='85%')
)

submit_button = widgets.Button(
    description='Run SOTA Ensemble',
    button_style='success',
    icon='search'
)

output_area = widgets.Output()

def on_button_clicked(b):
    with output_area:
        clear_output()
        full_text = text_input.value.strip()
        dist_text = span_input.value.strip()
        
        if not full_text:
            print("⚠️ Please enter some text first!")
            return
        
        print(f"🔍 Analyzing Statement...")
        print(f"• Full Text     : {full_text}")
        print(f"• Distorted Span: {dist_text if dist_text else '[None provided - using full text]'}\n")
        print("=" * 65)
        
        results = predict_with_aux_span(full_text, dist_text)
        
        for label, info in results.items():
            status = "🚨 DETECTED" if info["detected"] else "✅ Clear"
            print(f"• {label.replace('_', ' ').title().ljust(22)} : {status} "
                  f"(Prob: {info['probability']:.4f} | Thr: {info['threshold']})")
        print("=" * 65)

submit_button.on_click(on_button_clicked)

# Display the UI
display(widgets.VBox([text_input, span_input, submit_button, output_area]))

In [ ]:
# ================================================================
# CBT Cognitive Distortion Inference: Mental-RoBERTa + ModernBERT
# Five-fold probability ensembles + geometric-mean model ensemble
#
# IMPORTANT:
# 1. This expects five trained Mental-RoBERTa checkpoints:
#       ROBERTA_FOLD_DIR/fold_0/model.pt ... fold_4/model.pt
# 2. This expects five trained ModernBERT checkpoints:
#       MODERNBERT_FOLD_DIR/modernbert_fold_0/model.pt ... fold_4/model.pt
# 3. The provided Mental-RoBERTa training script does NOT save model.pt
#    checkpoints. You must export them during training or point this script
#    at actual saved checkpoints. Probability .npy files are not model weights.
# 4. THRESHOLD_JSON must contain the final calibrated threshold for each label.
# ================================================================

import os
import json
import warnings
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import ipywidgets as widgets
from IPython.display import display, clear_output
from transformers import AutoModel, AutoTokenizer
from peft import LoraConfig, get_peft_model, TaskType

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ------------------------- CONFIGURATION -------------------------

ROBERTA_BASE = (
    "/kaggle/input/models/arafahmed98/"
    "mental-roberta-stage2-distortion/transformers/default/1"
)
MODERNBERT_BASE = (
    "/kaggle/input/models/arafahmed99/strong-semantics-m/"
    "transformers/default/2/kaggle/working/modernbert-stage2-semantic"
)

# Update these if your trained checkpoint folders are stored elsewhere.
ROBERTA_FOLD_DIR = "/kaggle/working/new"
MODERNBERT_FOLD_DIR = "/kaggle/working/modernbert"

THRESHOLD_JSON = "/kaggle/working/final_therapy_platform_thresholds.json"

MAX_LEN = 512
DIST_LEN = 256
NUM_FOLDS = 5

MICRO_LABELS = [
    "all_or_nothing",
    "emotional_reasoning",
    "fortune_telling",
    "labeling",
    "magnification",
    "mental_filter",
    "mind_reading",
    "overgeneralization",
    "personalization",
    "should_statements",
]
NUM_LABELS = len(MICRO_LABELS)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Inference device: {device}")

# ------------------------- THRESHOLDS ----------------------------

if not os.path.isfile(THRESHOLD_JSON):
    raise FileNotFoundError(
        f"Threshold file not found: {THRESHOLD_JSON}\n"
        "Create/export the final calibrated thresholds JSON before inference."
    )

with open(THRESHOLD_JSON, "r", encoding="utf-8") as f:
    threshold_dict = json.load(f)

missing_labels = [label for label in MICRO_LABELS if label not in threshold_dict]
if missing_labels:
    raise ValueError(
        "Threshold JSON is missing labels: " + ", ".join(missing_labels)
    )

thresholds = np.asarray(
    [float(threshold_dict[label]) for label in MICRO_LABELS],
    dtype=np.float32,
)
if not np.all(np.isfinite(thresholds)) or np.any((thresholds < 0) | (thresholds > 1)):
    raise ValueError("All thresholds must be finite probabilities between 0 and 1.")

# ------------------------- TOKENIZERS ----------------------------

tokenizer_rob = AutoTokenizer.from_pretrained(
    ROBERTA_BASE, local_files_only=True
)
tokenizer_mod = AutoTokenizer.from_pretrained(
    MODERNBERT_BASE, local_files_only=True
)

# ================================================================
# MENTAL-ROBERTA ARCHITECTURE — copied from the training script
# ================================================================

class ContrastiveLoss(nn.Module):
    def __init__(self, temperature=0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, embeddings, labels):
        embeddings = F.normalize(embeddings, dim=-1)
        sim = torch.matmul(embeddings, embeddings.T) / self.temperature
        sim_exp = torch.exp(sim)
        labels = labels.float()
        pos_mask = (torch.matmul(labels, labels.T) > 0).float()
        positive = sim_exp * pos_mask
        denom = sim_exp.sum(dim=-1)
        loss = -torch.log((positive.sum(dim=-1) + 1e-8) / (denom + 1e-8))
        return loss.mean()


class RobertaBilinearGLUFusion(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.text_proj = nn.Linear(hidden_dim, hidden_dim)
        self.gate_proj = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, text_repr, local_repr):
        t = torch.tanh(self.text_proj(text_repr))
        g = torch.sigmoid(self.gate_proj(text_repr))
        return t * (g * local_repr)


class MultiLabelOnlyModel(nn.Module):
    def __init__(self):
        super().__init__()
        base_model = AutoModel.from_pretrained(ROBERTA_BASE)
        lora_config = LoraConfig(
            r=16,
            lora_alpha=32,
            target_modules=["query", "value"],
            lora_dropout=0.1,
            bias="none",
            task_type=TaskType.FEATURE_EXTRACTION,
        )
        self.encoder = get_peft_model(base_model, lora_config)

        for param in self.encoder.base_model.parameters():
            param.requires_grad = True

        hidden = self.encoder.base_model.config.hidden_size
        self.fusion = RobertaBilinearGLUFusion(hidden)
        self.expert_heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(hidden, hidden),
                nn.GELU(),
                nn.Dropout(0.2),
                nn.Linear(hidden, 1),
            )
            for _ in range(NUM_LABELS)
        ])
        self.prototypes = nn.Parameter(torch.randn(NUM_LABELS, hidden))
        self.multi_loss = nn.BCEWithLogitsLoss()
        self.contrastive = ContrastiveLoss()

    def masked_pool(self, hidden_states, token_mask):
        token_mask = token_mask.unsqueeze(-1)
        masked = hidden_states * token_mask
        denom = token_mask.sum(dim=1).clamp(min=1e-6)
        return masked.sum(dim=1) / denom

    def forward(
        self,
        input_ids,
        attention_mask,
        dist_input_ids,
        dist_attention_mask,
        token_mask,
        multi_labels=None,
    ):
        full_out = self.encoder(
            input_ids=input_ids, attention_mask=attention_mask
        )
        full_hidden = full_out.last_hidden_state
        full_repr = full_hidden[:, 0]

        dist_out = self.encoder(
            input_ids=dist_input_ids,
            attention_mask=dist_attention_mask,
        )
        dist_repr = dist_out.last_hidden_state[:, 0]
        masked_repr = self.masked_pool(full_hidden, token_mask)
        local_repr = (dist_repr + masked_repr) / 2.0
        fused_repr = self.fusion(full_repr, local_repr)

        outputs = [expert(fused_repr) for expert in self.expert_heads]
        multi_logits = torch.cat(outputs, dim=-1)
        proto_sim = torch.matmul(
            F.normalize(fused_repr, dim=-1),
            F.normalize(self.prototypes, dim=-1).T,
        )
        multi_logits = torch.clamp(multi_logits + proto_sim, -8.0, 8.0)
        return {"loss": None, "multi_logits": multi_logits}


# ================================================================
# MODERNBERT ARCHITECTURE — copied from the training script
# ================================================================

LABEL_SMOOTHING = 0.08
NUM_MACRO = 5


class SmoothedSqrtBCELoss(nn.Module):
    def __init__(self, class_weights, smoothing=0.05):
        super().__init__()
        self.register_buffer("class_weights", class_weights)
        self.smoothing = smoothing

    def forward(self, logits, targets):
        smoothed_targets = targets * (1.0 - self.smoothing) + 0.5 * self.smoothing
        loss_fct = nn.BCEWithLogitsLoss(reduction="none")
        return (
            loss_fct(logits, smoothed_targets)
            * self.class_weights.unsqueeze(0)
        ).mean()


class ModernBertBilinearGLUFusion(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.text_proj = nn.Linear(hidden_dim, hidden_dim)
        self.gate_proj = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, text_repr, local_repr):
        return torch.tanh(self.text_proj(text_repr)) * (
            torch.sigmoid(self.gate_proj(text_repr)) * local_repr
        )


class MultiSampleExpertHead(nn.Module):
    def __init__(self, hidden):
        super().__init__()
        self.dense = nn.Linear(hidden, hidden)
        self.act = nn.GELU()
        self.dropouts = nn.ModuleList([
            nn.Dropout(0.20 + 0.05 * i) for i in range(4)
        ])
        self.out = nn.Linear(hidden, 1)

    def forward(self, x):
        x = self.act(self.dense(x))
        if self.training:
            return torch.stack(
                [self.out(drop(x)) for drop in self.dropouts]
            ).mean(dim=0)
        return self.out(x)


class EarlyFusionSchemaModel(nn.Module):
    def __init__(self, class_weights):
        super().__init__()
        base_model = AutoModel.from_pretrained(MODERNBERT_BASE)
        lora_config = LoraConfig(
            r=32,
            lora_alpha=64,
            target_modules=["Wqkv", "Wo", "Wi"],
            lora_dropout=0.1,
            bias="none",
            task_type=TaskType.FEATURE_EXTRACTION,
        )
        self.encoder = get_peft_model(base_model, lora_config)

        for name, param in self.encoder.base_model.named_parameters():
            if "layers." in name:
                try:
                    param.requires_grad = (
                        int(name.split("layers.")[1].split(".")[0]) >= 16
                    )
                except ValueError:
                    param.requires_grad = True
            elif "embeddings" in name or "embed" in name:
                param.requires_grad = False
            else:
                param.requires_grad = True

        hidden = self.encoder.base_model.config.hidden_size
        self.fusion = ModernBertBilinearGLUFusion(hidden)
        self.macro_heads = nn.ModuleList([
            nn.Linear(hidden, 1) for _ in range(NUM_MACRO)
        ])
        self.expert_heads = nn.ModuleList([
            MultiSampleExpertHead(hidden) for _ in range(NUM_LABELS)
        ])
        self.register_buffer(
            "hierarchy_matrix",
            torch.tensor([
                [1, 0, 0, 0, 0],
                [0, 1, 0, 0, 0],
                [0, 0, 1, 0, 0],
                [0, 0, 0, 0, 1],
                [0, 0, 1, 0, 0],
                [1, 0, 0, 0, 0],
                [0, 0, 0, 1, 0],
                [1, 0, 0, 0, 0],
                [0, 0, 0, 1, 0],
                [0, 0, 0, 0, 1],
            ], dtype=torch.float32),
        )
        self.micro_loss = SmoothedSqrtBCELoss(
            class_weights=class_weights, smoothing=LABEL_SMOOTHING
        )
        self.macro_loss = nn.BCEWithLogitsLoss()

    def masked_pool(self, hidden_states, token_mask):
        token_mask = token_mask.unsqueeze(-1)
        return (hidden_states * token_mask).sum(dim=1) / (
            token_mask.sum(dim=1).clamp(min=1e-6)
        )

    def forward(
        self,
        input_ids,
        attention_mask,
        dist_input_ids,
        dist_attention_mask,
        token_mask,
        multi_labels=None,
    ):
        full_out = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            output_hidden_states=True,
        )
        dist_out = self.encoder(
            input_ids=dist_input_ids,
            attention_mask=dist_attention_mask,
        )
        full_hidden = full_out.last_hidden_state
        full_repr = full_hidden[:, 0]
        dist_repr = dist_out.last_hidden_state[:, 0]
        masked_repr = self.masked_pool(full_hidden, token_mask)
        local_repr = (dist_repr + masked_repr) / 2.0
        fused_repr = self.fusion(full_repr, local_repr)

        macro_logits = torch.cat(
            [head(fused_repr) for head in self.macro_heads], dim=-1
        )
        macro_probs = torch.sigmoid(macro_logits)
        structural_bias = torch.matmul(macro_probs, self.hierarchy_matrix.T)
        micro_logits = torch.cat(
            [expert(fused_repr) for expert in self.expert_heads], dim=-1
        )
        final_micro_logits = micro_logits + torch.log(structural_bias + 1e-8)
        return {"loss": None, "multi_logits": final_micro_logits}


# ------------------------- LOAD CHECKPOINTS ----------------------

def _load_state_dict(path):
    state = torch.load(path, map_location="cpu", weights_only=True)
    if isinstance(state, dict) and "state_dict" in state:
        state = state["state_dict"]
    # Be tolerant of checkpoints saved from a DDP-wrapped model.
    if isinstance(state, dict) and state and all(k.startswith("module.") for k in state):
        state = {k[len("module."):]: v for k, v in state.items()}
    return state


def _require_checkpoint(path, model_name):
    if not os.path.isfile(path):
        raise FileNotFoundError(
            f"{model_name} checkpoint not found:\n  {path}\n\n"
            "The inference code needs trained model state_dict checkpoints, "
            "not just tokenizer files or saved probability arrays."
        )


def load_roberta_folds():
    models = []
    for fold in range(NUM_FOLDS):
        path = os.path.join(ROBERTA_FOLD_DIR, f"fold_{fold}", "model.pt")
        _require_checkpoint(path, f"Mental-RoBERTa fold {fold}")
        model = MultiLabelOnlyModel()
        model.load_state_dict(_load_state_dict(path), strict=True)
        model.to(device).eval()
        models.append(model)
        print(f"Loaded Mental-RoBERTa fold {fold + 1}/{NUM_FOLDS}")
    return models


def load_modernbert_folds():
    models = []
    for fold in range(NUM_FOLDS):
        path = os.path.join(
            MODERNBERT_FOLD_DIR, f"modernbert_fold_{fold}", "model.pt"
        )
        _require_checkpoint(path, f"ModernBERT fold {fold}")
        model = EarlyFusionSchemaModel(
            class_weights=torch.ones(NUM_LABELS, dtype=torch.float32)
        )
        model.load_state_dict(_load_state_dict(path), strict=True)
        model.to(device).eval()
        models.append(model)
        print(f"Loaded ModernBERT fold {fold + 1}/{NUM_FOLDS}")
    return models


# ------------------------- SPAN MASKING --------------------------

def build_distortion_mask(full_text, distorted_text, tokenizer):
    """Reproduce the training scripts' token-subsequence span matching."""
    full_enc = tokenizer(
        full_text,
        add_special_tokens=False,
        truncation=True,
        max_length=MAX_LEN,
    )
    dist_enc = tokenizer(
        distorted_text,
        add_special_tokens=False,
        truncation=True,
        max_length=DIST_LEN,
    )
    full_ids = full_enc["input_ids"]
    dist_ids = dist_enc["input_ids"]

    mask = np.zeros(len(full_ids) + 2, dtype=np.float32)
    if len(dist_ids) == 0:
        return mask.tolist()

    for i in range(len(full_ids) - len(dist_ids) + 1):
        if full_ids[i:i + len(dist_ids)] == dist_ids:
            start = i + 1  # account for leading special token
            end = start + len(dist_ids)
            if end <= len(mask):
                mask[start:end] = 1.0
            break

    return mask.tolist()


def prepare_model_inputs(full_text, distorted_text, tokenizer):
    full_enc = tokenizer(
        full_text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_LEN,
        padding="max_length",
    )
    dist_enc = tokenizer(
        distorted_text,
        return_tensors="pt",
        truncation=True,
        max_length=DIST_LEN,
        padding="max_length",
    )

    mask = build_distortion_mask(full_text, distorted_text, tokenizer)
    # Model input has MAX_LEN positions. Match the training collator's
    # right-padding/right-truncation behavior for the auxiliary mask.
    mask = (mask + [0.0] * MAX_LEN)[:MAX_LEN]

    return {
        "input_ids": full_enc["input_ids"].to(device),
        "attention_mask": full_enc["attention_mask"].to(device),
        "dist_input_ids": dist_enc["input_ids"].to(device),
        "dist_attention_mask": dist_enc["attention_mask"].to(device),
        "token_mask": torch.tensor(
            [mask], dtype=torch.float32, device=device
        ),
    }


# ------------------------- PREDICTION ----------------------------

def _predict_fold_mean(models, inputs):
    fold_probs = []
    with torch.inference_mode():
        for model in models:
            logits = model(**inputs)["multi_logits"]
            probs = torch.sigmoid(logits).float().cpu().numpy()
            fold_probs.append(probs)
    return np.mean(np.stack(fold_probs, axis=0), axis=0)[0]


def predict_with_aux_span(full_text, distorted_text=""):
    """
    Returns per-label probability, calibrated threshold, and binary detection.
    The ensemble is:
      (1) arithmetic mean across 5 Mental-RoBERTa folds;
      (2) arithmetic mean across 5 ModernBERT folds;
      (3) geometric mean of the two model-family probabilities.
    """
    full_text = str(full_text).strip()
    distorted_text = str(distorted_text or "").strip()

    if not full_text:
        raise ValueError("Please enter a non-empty statement.")
    if not distorted_text:
        # Matches the behavior of the original interactive inference code.
        distorted_text = full_text

    rob_inputs = prepare_model_inputs(
        full_text, distorted_text, tokenizer_rob
    )
    mod_inputs = prepare_model_inputs(
        full_text, distorted_text, tokenizer_mod
    )

    probs_rob = _predict_fold_mean(model_rob_folds, rob_inputs)
    probs_mod = _predict_fold_mean(model_mod_folds, mod_inputs)

    # Both arrays have shape (NUM_LABELS,). Clip to avoid log(0).
    stacked = np.stack([probs_rob, probs_mod], axis=0)
    geo_probs = np.exp(
        np.mean(np.log(np.clip(stacked, 1e-12, 1.0)), axis=0)
    )

    predictions = (geo_probs >= thresholds).astype(np.int32)

    results = {}
    for i, label in enumerate(MICRO_LABELS):
        results[label] = {
            "probability": float(geo_probs[i]),
            "threshold": float(thresholds[i]),
            "detected": bool(predictions[i]),
            "roberta_probability": float(probs_rob[i]),
            "modernbert_probability": float(probs_mod[i]),
        }
    return results


# ------------------------- LOAD BOTH ENSEMBLES -------------------

# This will intentionally stop with a clear FileNotFoundError if a required
# checkpoint is missing. In the supplied Mental-RoBERTa training script,
# model.pt files were not saved, so export them before expecting this to run.
model_rob_folds = load_roberta_folds()
model_mod_folds = load_modernbert_folds()


# ------------------------- INTERACTIVE KAGGLE UI -----------------

text_input = widgets.Textarea(
    value=(
        "If I don't get a promotion this year, my entire career is "
        "an absolute failure."
    ),
    placeholder="Type a thought or statement here...",
    description="Full Text:",
    disabled=False,
    layout=widgets.Layout(width="95%", height="90px"),
)

span_input = widgets.Text(
    value="my entire career is an absolute failure",
    placeholder="Optional: enter the specific distorted phrase/span...",
    description="Distorted Span:",
    disabled=False,
    layout=widgets.Layout(width="95%"),
)

submit_button = widgets.Button(
    description="Run CBT Ensemble",
    button_style="success",
    icon="search",
)

output_area = widgets.Output()


def on_button_clicked(_):
    with output_area:
        clear_output()
        full_text = text_input.value.strip()
        dist_text = span_input.value.strip()

        if not full_text:
            print("Please enter a statement first.")
            return

        print("Analyzing statement...")
        print(f"Full text: {full_text}")
        print(
            "Distorted span: "
            + (dist_text if dist_text else "[not supplied; using full text]")
        )
        print("=" * 88)

        try:
            results = predict_with_aux_span(full_text, dist_text)
        except Exception as exc:
            print(f"Inference failed: {type(exc).__name__}: {exc}")
            return

        for label, info in results.items():
            status = "DETECTED" if info["detected"] else "Not detected"
            print(
                f"{label.replace('_', ' ').title():24s} | {status:13s} "
                f"| ensemble={info['probability']:.4f} "
                f"| threshold={info['threshold']:.4f} "
                f"| RoBERTa={info['roberta_probability']:.4f} "
                f"| ModernBERT={info['modernbert_probability']:.4f}"
            )
        print("=" * 88)
        detected = [
            label.replace("_", " ").title()
            for label, info in results.items()
            if info["detected"]
        ]
        print("Detected labels:", ", ".join(detected) if detected else "None")


submit_button.on_click(on_button_clicked)

display(
    widgets.VBox(
        [text_input, span_input, submit_button, output_area],
        layout=widgets.Layout(width="100%"),
    )
)
